# Zürich Tram Flow
**Verspätungsanalyse und Vorhersage im Tramnetz Zürich**

> **Datenbasis:** [`sf_data-research`](https://github.com/kaywiegand/sf_data-research) — Research & Data Engineering Phase (abgeschlossen)  
> **Erstellt mit:** [wgnd-scaffolding](https://github.com/kaywiegand/wgnd-scaffolding) · [wgnd-toolkit](https://github.com/kaywiegand/wgnd-toolkit)

## Facts



| Feld | Wert |
|------|------|
| **Business-Frage** | Wo, wann und warum entstehen Verspätungen im Zürcher Tramnetz — und lassen sie sich vorhersagen? |
| **Stakeholder** | VBZ (Betreiber), Stadtplanung Zürich, Fahrgäste |
| **Methode** | EDA → Korrelationsanalyse → Zeitreihen-/ML-Modell → Dashboard |
| **Hauptdatenquelle** | VBZ IST-Daten 2023–2025 (opentransportdata.swiss) + GTFS + Meteo + Events |
| **Ziel-Metrik** | Vorhersagegenauigkeit (MAE) pro Linie/Stadtkreis; On-Time Performance (OTP) |
| **Out of Scope** | Echtzeit-Feed (GTFS-RT), Daten ab Format v2 (ab Mitte 2025), VBB Berlin |
| **Analysezeitraum** | 2023–2025 (IST-Daten Format v1, einheitlich) |
| **Stack** | Python · Polars · Pandas · GeoPandas · Plotly · Folium |

## Context



### Scenario

Verspätungen im öffentlichen Nahverkehr sind ärgerlich — für Menschen und für das System.
Das Zürcher Tramnetz (VBZ) bietet eine außergewöhnlich gute Open-Data-Grundlage:
IST-Daten mit Echtzeit-Verspätungen pro Haltestelle, GTFS-Fahrplandaten, Wetterdaten
und Eventkalender — über drei Jahre (2023, 2024 und 2025).

Das Tram fährt im offenen Stadtverkehr — beeinflusst durch Autos, Fußgänger, Wetter,
Topografie und Großveranstaltungen. Das macht es zu einem besonders interessanten
Analysegegenstand für Betreiber, Stadtplanung und Fahrgäste.

### Mission

Aufbau einer vollständigen Analyse- und Vorhersage-Pipeline für Verspätungen im
Zürcher Tramnetz — vom validierten Master-Datensatz bis zum interaktiven Dashboard.
Zürich dient dabei als Referenzmodell für Städte wie Berlin, die ihre Datenpotenziale
noch nicht ausschöpfen.

### Zentrale Fragen

**Betrieb & Muster**
* Wo entstehen Verspätungen im Tramnetz — und zu welchen Zeiten?
* Welche Einflussfaktoren spielen die größte Rolle? (Wetter, Tageszeit, Events, Topografie)
* Lassen sich Verspätungen vorhersagen, bevor sie entstehen?

**Netz & Struktur**
* Hat der Netzausbau Dezember 2023 die Pünktlichkeit an den veränderten Linien und Stadtteilen verbessert oder verschlechtert?
* Zeigen neue Streckenabschnitte eine Einlaufzeit — performen sie in den ersten Monaten schlechter?
* Welche Knotenpunkte sind kritische Hotspots und lösen Kettenreaktionen aus?
* Welche Stadtteile haben durch den Ausbau mehr oder weniger Anbindung bekommen?

**Systemisch**
* Was kann ein Betreiber oder eine Stadt konkret besser machen?

### Methode & Metriken

| Metrik | Zielwert | Begründung |
|--------|----------|-------------|
| On-Time Performance (OTP) | Baseline messen | Anteil Fahrten < 2 Min Verspätung |
| Mean Absolute Error (MAE) | < 60 Sek | Vorhersagegenauigkeit pro Linie/Stadtkreis |
| Bottleneck Score | Top-10 Haltestellen | Haltestellen mit systemweiten Folgeverspätungen |
| Weather Sensitivity Score | Korrelation quantifizieren | Einfluss von Regen/Wind/Temperatur |
| Event Impact Score | Verspätungsanstieg messbar | Vergleich Event-Tage vs. normale Tage |

### Modellauswahl

> **TODO:** Dieser Abschnitt wird nach der Modellierungsphase ergänzt —
> finale Metriken, Hyperparameter und Validierungsergebnisse folgen in `03_analysis.ipynb`.

#### Modelle im Scope

| Modell | Rolle | Begründung |
|:---|:---|:---|
| **Ridge Regression** | Baseline | Wie viel erklären rein lineare Beziehungen? |
| **Random Forest** | Zwischenstufe | Nicht-lineare Interaktionen ohne intensives Tuning |
| **XGBoost** | Primärmodell | Goldstandard für tabellarisches Supervised Learning |
| **LightGBM** | Alternative | Schneller bei großen Datensätzen — Vergleichskandidat |

#### Empfohlene Reihenfolge

```
1. Ridge Regression   → Baseline
2. Random Forest      → Nicht-lineare Muster ohne viel Tuning
3. XGBoost            → Primärmodell
4. LightGBM           → Alternative bei langen Trainingszeiten
```

#### Warum keine klassischen Zeitreihenmodelle?

ARIMA und Prophet sind für univariate Zeitreihen ausgelegt — hier haben wir tausende parallele Trips
mit expliziten Zeit-Features (`hour`, `is_weekend`, `season` etc.).
Die Zeitdimension steckt bereits in den Features → kein Sequenzmodell nötig.

## Netzstruktur

Das Zürcher Tramnetz (VBZ) besteht im Analysezeitraum 2023–2025 aus **16–18 Linien** je nach Fahrplanjahr.
Die GTFS-Daten werden jährlich als neue Version veröffentlicht — **j23**, **j24** und **j25** entsprechen den drei Betriebsjahren.

> **Interaktive Karte:** [`reports/figures/tram_lines_map.html`](../reports/figures/tram_lines_map.html)
> Alle Linien mit offiziellen VBZ-Farben, Haltestellennamen und Streckenvergleich 2023 / 2024 / 2025.
> Linien und Jahre können einzeln ein- und ausgeblendet werden.

### Fahrplanwechsel Dezember 2023 — j23 → j24

Der Fahrplanwechsel im Dezember 2023 war der **größte Netzausbau in der Geschichte der VBZ**
(*Tramnetz Süd*). Drei Linien wurden fundamental umgebaut:

| Linie | j23 Halte | j24 Halte | Veränderung | Neue Abschnitte |
| :---: | ---: | ---: | :--- | :--- |
| **9** | 24 | 32 | +8 Halte | Bellevue · Paradeplatz · Sihlstrasse · Goldbrunnenplatz |
| **11** | 20 | 33 | +13 Halte | Stadelhofen · Kreuzplatz · Burgwies · Rehalp |
| **13** | 11 | 30 | +19 Halte (+173%) | Altstetten · HB · Paradeplatz · Enge · Sihlcity Nord |
| **7** | 31 | 31 | 2 Halte umbenannt | Post Wollishofen → Renggerstrasse |
| **15** | 13 | 13 | 1 Halt umbenannt | Bucheggplatz → Bucheggplatz D |

Linien **10, 12, 14, 17** sind über alle drei Jahre identisch.
Linie **18** existiert nur im Fahrplanjahr 2024 (j24).

### Implikationen für die Analyse

- **Linienvergleiche über Zeit:** Linien 9, 11 und 13 sind in j23 strukturell andere Linien als in j24/j25 — kürzere Strecken, weniger Halte, anderes Betriebsmuster. Direkter Jahresvergleich für diese Linien ist mit Vorsicht zu interpretieren.
- **Cancellation-Raten:** Erhöhte Ausfallraten in 2023 bei mehreren Linien können teilweise auf kürzere/andere Streckenführungen zurückzuführen sein — nicht zwingend auf schlechtere Betriebsqualität.
- **Feature Engineering:** `line_name` allein reicht nicht — das GTFS-Jahr (j23 vs. j24/j25) ist ein implizites Kontextmerkmal, das strukturelle Unterschiede kodiert.
- **`canceled`-Flag:** Netzweit erhöhte Rate Jan 2023 – Jun 2024, simultane Normalisierung Juli 2024 — wahrscheinlich eine Datendefinitions-Änderung beim Provider (opentransportdata.swiss), nicht ein Infrastrukturproblem. Siehe Finding F-TARGET-05.

## Data




Die gesamte Data-Engineering-Phase wurde in einem separaten Research-Repo durchgeführt
und ist dort vollständig dokumentiert:

> **Quelle:** [`sf_data-research`](https://github.com/kaywiegand/sf_data-research)  
> **Status:** Phase 1 abgeschlossen — Datenbasis vollständig und validiert.

### Was wurde dort gemacht?

| Schritt | Beschreibung | Notebook |
| :--- | :--- | :--- |
| IST-Daten | Download 36 ZIP-Archive (38 GB), Filter auf VBZ & Tram, Parquet-Konvertierung | `vbz-ist-daten.ipynb` |
| GTFS | Fahrplandaten 2023–2025, Spatial Join Stadtkreise, Haltestellen-Lookup | `vbz-gtfs-data.ipynb` |
| Meteo | 3 Quellen konsolidiert (Stampfenbachstr. + Mythenquai), Stundenmittelwerte | `vbz-meteo-data.ipynb` |
| Events | 301 Einträge, 5 Kategorien, Gewichtungsschema 1–3 | `vbz-events-data.ipynb` |
| Benchmark | Polars vs. Pandas: 4× schneller, 4× weniger RAM | `vbz-pandas-vs-polars.ipynb` |
| Master-Merge | Left Join IST + GTFS + Meteo + Events → `vbz_master.parquet` | `vbz-data-master-preparation.ipynb` |
| Validierung | 8 Checks: Schema, Abdeckung, Wertebereiche, Nulls, Join-Qualität, Business-Logik | `vbz-data-master-validation.ipynb` |

### Wichtige Entscheidungen aus der Research-Phase

| Entscheidung | Was | Warum |
| :--- | :--- | :--- |
| Polars statt Pandas | Haupt-DataFrame-Bibliothek | 4× schneller, 4× weniger RAM bei 94 Mio. Zeilen |
| Left Join überall | Merge-Strategie | Kein Datenverlust durch Join-Lücken |
| 2024 als GTFS-Referenzjahr | Fahrplandaten | Vollständigste Datenlage, stabilstes Jahr |
| 2 Meteo-Stationen | Stampfenbachstrasse + Mythenquai | Zwei Topografien: Stadtlage vs. Seelage |
| Scope 2023–2025 v1 | Analysezeitraum | Einheitliches Datenformat, kein Mischformat |
| Stadtkreis im Lookup | district im GTFS-Join | Einmalig sauber im Master, kein wiederholter Spatial Join |
| Ausfälle behalten | `canceled = True` | Extremster Verspätungsfall, für Modell unverzichtbar |
| Schwellenwert Events | >1.000 Besucher | Kleinere Events kein messbarer Netzeinfluss |
| `trip_id` + `stop_sequence` | GTFS-Join-Erweiterung | Trip-Level-Analysen, Kaskadeneffekte, Hotspot-Erkennung |

### Datenmenge

| Stufe | Menge |
| :--- | :--- |
| Rohdaten (schweizweit, komprimiert) | ~38 GB (36 ZIP-Archive) |
| Rohdaten entpackt | ~500–720 GB |
| Nach Filter VBZ + Tram (Parquet) | ~1,44 GB (1.096 Dateien) |
| Master-Datensatz | ~567 MB · 94 Mio. Zeilen · 26 Spalten |

### Data Dictionary



**Datei:** `data/raw/zh-tram-data-master.parquet`  
**Zeilen:** ~94 Millionen · **Spalten:** 26 · **Zeitraum:** 2023–2025

#### IST-Daten (Verkehr)

| # | Spaltenname | Typ | Beschreibung |
| :--- | :--- | :--- | :--- |
| 1 | `operating_date` | `Date` | Betriebstag |
| 2 | `line_name` | `Categorical` | Tramliniennummer (z.B. `"11"`) |
| 3 | `bpuic` | `Int32` | Haltestellen-ID — Join-Schlüssel zu GTFS |
| 4 | `arrival_schedule` | `Datetime` | Planmäßige Ankunftszeit |
| 5 | `arrival_delay` | `Float32` | Verspätung Ankunft in **Sekunden** (negativ = zu früh) |
| 6 | `departure_schedule` | `Datetime` | Planmäßige Abfahrtszeit |
| 7 | `departure_delay` | `Float32` | Verspätung Abfahrt in **Sekunden** |
| 8 | `canceled` | `Boolean` | Fahrtausfall = `True` — Quelle: `FAELLT_AUS_TF` (opentransportdata.swiss) |
| 9 | `trip_id` | `Categorical` | Fahrt-ID aus GTFS — Schlüssel für Trip-Level-Analysen (Kaskaden, Hotspots) |
| 10 | `stop_sequence` | `Int32` | Position des Halts innerhalb der Fahrt (1 = erster Halt) |

> ⚠️ **`canceled` — Datendefinitions-Änderung beim Provider:** Die Ausfallrate ist netzweit erhöht von Jan 2023 bis Jun 2024 und normalisiert sich simultan im Juli 2024 bei allen Linien gleichzeitig. Wahrscheinliche Ursache: opentransportdata.swiss hat `FAELLT_AUS_TF` bis Jun 2024 auch für Teilausfälle (Kurzwendungen) gesetzt — ab Jul 2024 nur noch für vollständige Fahrtausfälle. **Konsequenz für das Modell:** `canceled = True` aus dem Delay-Regressionsmodell ausschließen; `is_pre_july_2024` als Feature für ein separates Cancellation-Modell. → Finding F-TARGET-05

#### GTFS (Fahrplan & Geodaten)

| # | Spaltenname | Typ | Beschreibung |
| :--- | :--- | :--- | :--- |
| 11 | `stop_name` | `Categorical` | Haltestellenname (z.B. `"Paradeplatz"`) |
| 12 | `stop_lat` | `Float32` | Breitengrad (WGS84) |
| 13 | `stop_lon` | `Float32` | Längengrad (WGS84) |
| 14 | `district_nr` | `Int8` | Stadtkreis 1–12 (`null` = außerhalb Stadtgebiet) |
| 15 | `district_name` | `Categorical` | Stadtkreisname (z.B. `"Kreis 1"`) |

#### Meteo-Daten (Wetter)

| # | Spaltenname | Typ | Beschreibung |
| :--- | :--- | :--- | :--- |
| 16 | `temperature` | `Float32` | Temperatur in °C |
| 17 | `humidity` | `Float32` | Relative Luftfeuchtigkeit in % |
| 18 | `rain_duration` | `Float32` | Regendauer in min/h |
| 19 | `precipitation` | `Float32` | Niederschlagsmenge in mm |
| 20 | `wind_speed` | `Float32` | Windgeschwindigkeit in km/h |
| 21 | `global_radiation` | `Float32` | Globalstrahlung in W/m² |
| 22 | `flood_intensity` | `Int16` | Überschwemmungsindikator (ERZ-Meldungen) |

#### Event-Daten

| # | Spaltenname | Typ | Beschreibung |
| :--- | :--- | :--- | :--- |
| 23 | `event_name` | `Categorical` | Name des Events (`null` = kein Event an diesem Tag) |
| 24 | `event_type` | `Categorical` | Kategorie: `Feiertag`, `Stadtfest`, `Konzert`, `Messe`, `Fussball` |
| 25 | `event_size` | `Int8` | Gewichtung: `1` = mittel (>1k), `2` = groß (10k–30k), `3` = sehr groß (>30k) |
| 26 | `event_location` | `Categorical` | Veranstaltungsort (`null` = kein Event) |

#### Join-Strategie

| Join | Schlüssel | Typ |
| :--- | :--- | :--- |
| IST + GTFS Stops | `bpuic` = `bpuic` | Left Join |
| IST + Meteo | `floor(arrival_schedule, '1h')` = `date_time` | Left Join |
| IST + Events | `date(operating_date)` = `Datum` | Left Join |

> **Left Join überall:** Jede Tram-Fahrt bleibt im Datensatz erhalten. Fehlende Werte (z.B. Haltestellen außerhalb Stadtgebiet, Stunden ohne Wetterdaten) erscheinen als `null`.


### GTFS-Referenztabellen

**Verzeichnis:** `data/raw/gtfs/` — Referenzjahr 2024 (vollständigste Datenlage)

| Datei | Beschreibung | Verwendung |
| :--- | :--- | :--- |
| `gtfs_stops_lookup.parquet` | Haltestellen-Lookup: `bpuic` → `stop_name`, Koordinaten, `district_nr`, `district_name` | Join-Tabelle im Master |
| `gtfs_tram_stops.parquet` | Alle VBZ-Tram-Haltestellen mit Koordinaten | Geo-Visualisierungen |
| `gtfs_tram_routes.parquet` | Tramlinien (Route-ID, Linienname, Farbe) | Linien-Visualisierungen |
| `gtfs_tram_shapes.parquet` | Tram-Streckenverläufe als Koordinaten-Sequenzen | Streckenkarte |
| `gtfs_tram_trips.parquet` | Fahrten (Trip-ID, Route-ID, Shape-ID) | Verknüpfung Fahrten ↔ Strecken |
| `gtfs_zurich_stops.parquet` | Alle Zürich-Haltestellen (ZVV, nicht nur Tram) | Gesamtnetz-Überblick |
| `gtfs_zurich_routes.parquet` | Alle ZVV-Linien | Gesamtnetz-Überblick |
| `gtfs_zurich_shapes.parquet` | Alle ZVV-Streckenverläufe | Gesamtnetz-Karte |
| `gtfs_zurich_trips.parquet` | Alle ZVV-Fahrten | Gesamtnetz-Überblick |

## Workflow



### Phasen

| Phase | 00 Introduction | 01 Exploration | 02 Preparation | 03 Analysis | 04 Insights |
|-------|-----------------|----------------|----------------|-------------|-------------|
| | Projektkontext | Data Profiling | Feature Engineering | Modellierung | Reporting |
| | Data Dictionary | Verteilungen | Cleaning & Transformation | Training & Eval | Visualisierungen |
| | Datenbeschreibung | Erste Hypothesen | Outlier Handling | Vorhersagen | Executive Summary |
| | Quellen & Entscheidungen | Korrelationen | Split-Strategie | Validierung | Dashboard-Vorbereitung |

### Konventionen

#### Variablen-Präfix

| Präfix | Typ | Bedeutung |
|:---|:---|:---|
| `lf_` | `pl.LazyFrame` | Noch nicht im RAM — Operationen werden zu einem Scan zusammengefasst |
| `df_` | `pl.DataFrame` | Nach `.collect()` — vollständig im RAM |

> **Regel:** `lf_` solange die Pipeline aufgebaut wird. Einmalig `.collect()` → ab dann `df_`.

---

#### Variablen über alle Notebooks

`lf_raw` ist der gemeinsame Startpunkt — gleicher Name in EDA und Preparation.

| Variable | Notebook | Herkunft |
|:---|:---|:---|
| `lf_raw` | EDA · Preparation | `pl.scan_parquet(master)` — kein RAM-Verbrauch |
| `df_eda` | EDA | `lf_raw.gather_every(n).collect()` — Sample für Exploration (~1 Mio. Zeilen) |
| `df_delays_clean` | EDA | `df_eda` gefiltert (Extremverspätungen > 3600s entfernt) |
| `lf_clean` | Preparation | `structural_cleaning_pipeline(lf_raw)` — lazy, noch nicht collected |
| `df_clean` | Preparation | `lf_clean.collect()` — nach Phase 1, einmalig im RAM |
| `df_train` / `df_test` | Preparation | Temporal Split aus `df_clean` (2023–2024 / 2025) |
| `df_train_prep` / `df_test_prep` | Preparation | Nach Meteo-Imputation (Phase 3) |
| `df_train_feat` / `df_test_feat` | Preparation | Nach Feature Engineering (Phase 4) |


### Setup

In [ ]:
import polars as pl
import pandas as pd
from pathlib import Path

from zh_tram_flow.config import PATHS
from zh_tram_flow.settings import setup_plotting, logger

setup_plotting()
logger.info("00_introduction.ipynb gestartet")


### Dateicheck

In [ ]:
master_path = PATHS["raw"] / "zh-tram-data-master.parquet"

# Schema prüfen ohne vollständiges Laden
df_schema = pl.read_parquet(master_path, n_rows=1)

print(f"Datei: {master_path}")
print(f"Spalten: {len(df_schema.columns)}")
print()
print(df_schema.schema)

In [ ]:
# Erste Zeilen anschauen
df_sample = pl.read_parquet(master_path, n_rows=5)
df_sample

In [ ]:
# Zeilenanzahl (lazy, ohne alles in RAM zu laden)
row_count = pl.scan_parquet(master_path).select(pl.len()).collect().item()
print(f"Gesamtzeilen: {row_count:,}")

In [ ]:
# GTFS-Referenztabellen prüfen
gtfs_path = PATHS["raw"] / "gtfs"

for f in sorted(gtfs_path.glob("*.parquet")):
    df_tmp = pl.read_parquet(f, n_rows=1)
    print(f"{f.name}: {len(df_tmp.columns)} Spalten")